# LLM Laika
Antes de rodar: menu **Ambiente de execução → Alterar tipo → GPU (T4)**.

In [ ]:
# 1) Clonar/atualizar, ligar o Google Drive e baixar os dados
#    - 400 livros do Gutenberg + 30 mil artigos da Wikipédia (português moderno)
# Tudo fica no seu Drive (pasta LLM_Laika), então sobrevive se a sessão cair.
# Pode rodar de novo à vontade: não apaga nada e não baixa de novo o que já existe.
import os, torch
print("GPU disponível:", torch.cuda.is_available())

from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/LLM_Laika"

os.chdir("/content")
if not os.path.exists("LLM_Laika"):
    !git clone https://github.com/BrunoDevPro/LLM_Laika.git
%cd /content/LLM_Laika
!git pull -q
!pip install -q tokenizers datasets

# Pastas grandes viram atalhos para o Drive
for local, remote in [("data/books", "books"), ("data/wiki", "wiki"),
                      ("data/qa_auto", "qa_auto"), ("checkpoints", "checkpoints")]:
    remote = f"{DRIVE}/{remote}"
    os.makedirs(remote, exist_ok=True)
    if not os.path.islink(local):
        if os.path.isdir(local) and os.listdir(local):
            !cp -rn {local}/. {remote}/
        !rm -rf {local}
        os.symlink(remote, local)

!python prepare_data.py --max-books 400 --wiki 30000

In [ ]:
# 2) Pré-treino (aprende português) + fine-tuning (aprende a responder)
# O pré-treino só roda se ainda não existir um feito com os dados novos (livros + Wikipédia).
# Para forçar um pré-treino novo, apague checkpoints/pretrain_v2.ok no Drive.
import os
if os.path.exists("checkpoints/pretrain_v2.ok"):
    print("Pré-treino com os dados novos já existe no Drive: pulando.")
else:
    !python train.py --preset colab && touch checkpoints/pretrain_v2.ok
!python finetune.py

In [ ]:
# 3) Conversar (digite 'sair' para encerrar). Para testar só o pré-treino: sys.argv = ["chat.py", "--base"]
import sys
sys.argv = ["chat.py"]
%run chat.py